# MiniMax H3 weights · diffusion

Run this notebook with **Accelerator: None**, **Internet ON**. It downloads only the **diffusion** group into `/kaggle/working/models` and verifies each pinned SHA-256. It uses only Python's standard library: no CUDA, pip, virtualenv or GPU time.

After success, **Save Version** with outputs. Give this notebook a distinct title, such as `H3 weights diffusion`. In the rendering notebook, use **Add Input → Your Work** to attach this saved notebook output. Repeat for the other two preparation notebooks. Keep model outputs private unless the model licenses permit redistribution.

The three groups are separated to keep each below 20 GiB. Actual free space is checked before downloading. Diffusion alone needs 19.53 GiB; if this session has less, use a larger preparation environment or a Kaggle dataset containing the same pinned files. The main notebook can also download directly to `/kaggle/temp` if that volume has enough room.

Choose `MODEL_FAMILY='fl2va'` in all preparation/rendering notebooks if you want FL2VA instead of the original Ref2VA family.

In [ ]:
from pathlib import Path
import json,shutil
MODEL_FAMILY='ref2va'
SHARD='diffusion'
OUTPUT=Path('/kaggle/working/models')
ASSETS=[{'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'family': 'ref2va', 'size': 20970379616, 'sha256': '9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors', 'family': 'ref2va', 'size': 1956193000, 'sha256': '5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'family': 'fl2va', 'size': 20970379616, 'sha256': 'e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors', 'family': 'fl2va', 'size': 1956192992, 'sha256': 'c396a9a06f58399e9df9754b18299818d84a2ddd371724ba48fe4a41221437dc'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'size': 15687142551, 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'vae/minimax_h3_video_vae_fp16.safetensors', 'size': 5207808496, 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522'}, {'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': 'vae/minimax_h3_audio_vae_fp32.safetensors', 'size': 605254808, 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48'}, {'repo': 'speach1sdef178/MiniMax-H3-Semantic-Bridge', 'revision': 'b9fe58ba6f428d990a59f20f09f719c8fbc67f7d', 'file': 'MiniMaxH3_SemanticBridge_v1.safetensors', 'dest': 'semantic_bridge/MiniMaxH3_SemanticBridge_v1.safetensors', 'size': 11023032, 'sha256': 'ac0dc8ac05f545ebdee12e2fcebe4515b049f9cfd9558eb4887a9bf3fd6d562e'}, {'repo': 'speach1sdef178/MiniMax-H3-Semantic-Bridge', 'revision': 'b9fe58ba6f428d990a59f20f09f719c8fbc67f7d', 'file': 'MiniMax_H3_Semantic_Bridge_v1.0.zip', 'dest': None, 'size': 4406, 'sha256': 'c4866a0cc554237046814c9cb827a5960caecbd2214d5a5a04b0e43716b437bb'}, {'repo': 'barelymining/ComfyUI-MiniMax-H3-FastVideo', 'revision': 'aa554050fe27ec3c62e20e57519beb46741c9000', 'file': 'fasth3_vsa_gate.safetensors', 'dest': 'loras/fasth3_vsa_gate.safetensors', 'size': 3853522616, 'sha256': 'bf95408e335c4b0d6a1a44946a428a3e12849e6eab2de9566ecedae4a0043420', 'vsa': True}]
assert MODEL_FAMILY in {'ref2va','fl2va'}
selected=[a for a in ASSETS if (not a.get('family') or a['family']==MODEL_FAMILY)
          and a['repo']=='Comfy-Org/MiniMax-H3'
          and (a['file'].startswith('diffusion_models/') if SHARD=='diffusion' else
               a['file'].startswith('text_encoders/') if SHARD=='encoder' else
               a['file'].startswith(('vae/','loras/')))]
OUTPUT.mkdir(parents=True,exist_ok=True)
remaining=0
for a in selected:
    target=OUTPUT/a['file'];partial=target.with_suffix(target.suffix+'.part')
    if target.is_file():continue  # Complete files are checksum-verified below.
    remaining+=max(0,a['size']-(partial.stat().st_size if partial.exists() else 0))
free=shutil.disk_usage(OUTPUT).free
print(f'Payload: {sum(a["size"] for a in selected)/2**30:.2f} GiB; remaining: {remaining/2**30:.2f} GiB; free: {free/2**30:.2f} GiB')
assert remaining+64*2**20<=free,'Insufficient preparation disk; select a session with more free space.'


In [ ]:
def download_verified(asset,root):
    """Stream a pinned file, resume partial downloads, and verify before publishing."""
    from urllib.request import Request,urlopen
    from urllib.error import URLError,HTTPError
    from http.client import IncompleteRead
    import hashlib,time
    destination=root/asset['file']
    destination.parent.mkdir(parents=True,exist_ok=True)
    def digest(path):
        h=hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
        return h.hexdigest()
    if destination.exists():
        assert destination.stat().st_size==asset['size'] and digest(destination)==asset['sha256'],str(destination)+' is invalid'
        print('Already verified:',destination.name);return destination
    partial=destination.with_suffix(destination.suffix+'.part')
    url=f"https://huggingface.co/{asset['repo']}/resolve/{asset['revision']}/{asset['file']}?download=true"
    for attempt in range(8):
        offset=partial.stat().st_size if partial.exists() else 0
        assert offset<=asset['size'],'Partial file exceeds expected size: '+str(partial)
        if offset==asset['size']:break
        headers={'Range':f'bytes={offset}-'} if offset else {}
        try:
            with urlopen(Request(url,headers=headers),timeout=60) as response:
                if offset and response.status!=206:
                    offset=0  # Server ignored Range; restart without appending duplicate bytes.
                if response.status==206:
                    assert response.headers.get('Content-Range','').startswith(f'bytes {offset}-'),'Unexpected range response'
                written=offset;last_report=time.monotonic()
                with partial.open('ab' if offset else 'wb') as stream:
                    while block:=response.read(8*1024**2):
                        if written+len(block)>asset['size']:raise ValueError('Server sent too many bytes')
                        stream.write(block);written+=len(block)
                        if time.monotonic()-last_report>30:
                            print(f'{destination.name}: {written/2**30:.2f}/{asset["size"]/2**30:.2f} GiB',flush=True)
                            last_report=time.monotonic()
            if partial.stat().st_size!=asset['size']:raise OSError('Incomplete download')
            break
        except (OSError,URLError,IncompleteRead) as error:
            if attempt==7:raise RuntimeError('Download failed; rerun to resume '+str(partial)) from error
            print('Retrying resumable download:',destination.name,type(error).__name__,flush=True)
            time.sleep(min(2**attempt,30))
    assert partial.stat().st_size==asset['size'] and digest(partial)==asset['sha256'],'SHA-256 verification failed: '+str(partial)
    partial.replace(destination)  # Same volume; no second full-size copy.
    print('Verified:',destination.name,flush=True)
    return destination


In [ ]:
for asset in selected:download_verified(asset,OUTPUT)
(OUTPUT.parent/('model-manifest-'+SHARD+'.json')).write_text(json.dumps(selected,indent=2))
print('Finished. Save Version with outputs, then attach this notebook output to the rendering notebook.')